# Install labor_cost from GitLab

Use your company's virtual-environment procedure with
`production/runtime/requirements.txt`, then select that environment's
kernel. Python 3.10+ is required; use a version supported by the pinned
NumPy/SHAP dependencies (Python 3.10–3.12 is the initial target).

Run the configuration and installation cells first. **Restart the kernel**
after installation, then run only the verification cells below the restart
marker. Do not use Run All to perform an upgrade in an already-used kernel.
The GitLab bootstrap and package must come from a repository you trust.
See [the deployment guide](../docs/gitlab-deployment.md).


In [ ]:
import sys

GITLAB_API = "https://gitlab.example.com/api/v4"
GITLAB_PROJECT = "your-group/driver-model"  # Or a numeric project ID.
GITLAB_REF = "main"  # A release tag/full SHA is preferable for repeat runs.
PACKAGE_SUBDIR = "production"  # Use "." if production/ is the repo root.

# Z_ENV_DAA_GITLAB_TOKEN is supplied by your existing secret mechanism.
# The installation cell also accepts an environment variable of that name.
print("Notebook Python:", sys.executable)
print("Python version:", sys.version.split()[0])


In [ ]:
import json
import os
import re
from urllib.parse import quote, urlsplit

import requests


def gitlab_read(session, endpoint, **params):
    """Read this GitLab project without redirects or credential logging."""
    url = (
        GITLAB_API.rstrip("/") + "/projects/"
        + quote(str(GITLAB_PROJECT), safe="") + "/" + endpoint
    )
    try:
        response = session.get(
            url, params=params, timeout=(15, 120), allow_redirects=False,
        )
    except requests.RequestException:
        raise RuntimeError(
            "GitLab request failed; check network and corporate CA setup."
        ) from None
    if response.status_code != 200:
        status = response.status_code
        response.close()
        raise RuntimeError(
            f"GitLab returned HTTP {status}; check URL, project, ref and "
            "token permissions. Redirects are not followed."
        )
    return response


parsed_api = urlsplit(GITLAB_API)
if (
    parsed_api.scheme != "https" or not parsed_api.hostname
    or parsed_api.username is not None or parsed_api.password is not None
    or parsed_api.query or parsed_api.fragment
):
    raise ValueError("Use an HTTPS API URL without credentials or query.")
if parsed_api.hostname == "gitlab.example.com":
    raise ValueError("Configure your GitLab URL, project and ref first.")
if sys.version_info < (3, 10):
    raise RuntimeError("Select a Python 3.10+ kernel before installing.")

with requests.Session() as gitlab_session:
    gitlab_session.headers["PRIVATE-TOKEN"] = (
        globals().get("Z_ENV_DAA_GITLAB_TOKEN")
        or os.environ.get("Z_ENV_DAA_GITLAB_TOKEN")
    )
    try:
        if not gitlab_session.headers["PRIVATE-TOKEN"]:
            raise RuntimeError("Provide Z_ENV_DAA_GITLAB_TOKEN via secrets.")
        with gitlab_read(
            gitlab_session,
            "repository/commits/" + quote(GITLAB_REF, safe=""),
        ) as response:
            CODE_COMMIT = response.json()["id"]
        if not re.fullmatch(r"[0-9a-fA-F]{40}|[0-9a-fA-F]{64}", CODE_COMMIT):
            raise RuntimeError("GitLab did not return a full commit SHA.")
        bootstrap_path = "/".join(
            part for part in (PACKAGE_SUBDIR.strip("/"),
                              "runtime/bootstrap.py") if part != "."
        )
        with gitlab_read(
            gitlab_session,
            "repository/files/" + quote(bootstrap_path, safe="") + "/raw",
            ref=CODE_COMMIT,
        ) as response:
            loader_source = response.text
        loader = {"__name__": "labor_cost_gitlab_bootstrap"}
        exec(compile(loader_source, "GitLab/bootstrap.py", "exec"), loader)
        installation = loader["install_project"](
            GITLAB_API, GITLAB_PROJECT, CODE_COMMIT, gitlab_session,
            package_subdir=PACKAGE_SUBDIR,
        )
    finally:
        gitlab_session.headers.clear()

print("Requested ref:", GITLAB_REF)
print(json.dumps(installation, indent=2))
print("Restart the kernel, then run the verification cells below.")


## Restart the kernel here

Save the displayed commit and interpreter in your deployment/run notes.
Use Jupyter's **Restart Kernel**, then continue below; do not rerun the
installation cells unless you intend to reinstall. Restarting clears
previously imported package and dependency versions. The cells below do
not require the token, GitLab connectivity, or the repository checkout.


In [ ]:
import sys
from importlib.metadata import version

import labor_cost

print("Python:", sys.executable)
print("Installed distribution:", version("labor-cost-model"))
print("Imported library:", labor_cost.__file__)
# Expect the selected environment's site-packages, not a local src/ tree.


In [ ]:
import pandas as pd

from labor_cost import (
    AnalysisRequest, DatasetSnapshot, Factor, FieldSpec, Game, ModelGraph,
    ModelSpec, RowCost, SourceSchema, Switch, analyze, build_state,
    compile_model, cost, field, local_game, organization_game,
    outer_factors, value,
)

# A tiny signed-payment model for deployment verification only.
# It does not replace your analyst-authored payroll model.
model = ModelSpec(
    "deployment-smoke", "1",
    graph=ModelGraph(
        source=SourceSchema(), populations=(),
        fields=(FieldSpec("amount", "actual"),),
        row_costs=(RowCost("all", field("amount")),),
        factors=outer_factors() + (
            Factor("payment", "payment", formula=field("amount")),
        ),
        games=(Game("unit", value("payment"), (Switch("payment"),)),),
        team_game="unit",
        organization=organization_game("unit"),
        local=local_game("unit"),
    ),
)
compiled = compile_model(model)
rows = pd.DataFrame({
    "period": ["2026-08-01", "2026-09-01"],
    "employee_id": ["smoke-employee", "smoke-employee"],
    "employment_type": ["permanent", "permanent"],
    "team": ["Smoke", "Smoke"],
    "actual": [-100.0, 150.0],
})
dataset = DatasetSnapshot("deployment-smoke", "1", rows)
state = build_state(
    compiled, dataset, "2026-08", source_cost_column="actual",
)
assert abs(cost(state, compiled) - (-100.0)) < 1e-6
result = analyze(
    compiled, dataset,
    AnalysisRequest("2026-08", "2026-09", source_cost_column="actual"),
)
assert result.table("validation").passed.all()
assert abs(result.table("driver_bridge").impact.sum() - 250.0) < 1e-6
print("Deployment smoke passed: -100 -> 150; reconciled change +250.")
result.table("driver_bridge")


After this check, use your published model and real data through the normal
library API. Installation does not load payroll or publish a model/catalog.
The existing analytical gaps remain tracked in
[architecture-gaps.md](../docs/architecture-gaps.md).
